<a href="https://colab.research.google.com/github/sanghiteja244-pixel/Pharma_AI_Orchestrator/blob/main/PRI_PYTHON.Ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. INSTALL LATEST LIBRARIES
!pip install -U crewai crewai_tools langchain_groq
!pip install litellm
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, LLM, Process
from crewai_tools import SerperDevTool

# 2. CONFIGURE API KEYS (Ensure these are in your Colab Secrets 🔑)
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
groq_key = userdata.get('GROQ_API_KEY')

# 3. SETUP THE BRAIN (Groq Llama 3.3)
# Llama 3.3 70B is currently the fastest and most reliable for pharma agents.
groq_llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=groq_key,
    temperature=0.1 # Keeps the output factual and accurate
)

# 4. SETUP THE EYES (Regulatory Search)
search_tool = SerperDevTool(
    description="Targeted search for ClinicalTrials.gov and FDA.gov data."
)

# 5. DEFINE THE PHARMA AGENT TEAM
researcher = Agent(
    role='Lead Regulatory Researcher',
    goal='Extract updates for {drug_name} related to {topic} ONLY from ClinicalTrials.gov and FDA.gov.',
    backstory='Specialist in Pharmacy Regulatory Affairs. You verify data only through primary government sources.',
    tools=[search_tool],
    llm=groq_llm,
    verbose=True
)

analyst = Agent(
    role='Pharma Risk Analyst',
    goal='Identify specific clinical or market risks for {drug_name} based on latest research.',
    backstory='Expert in pharmaceutical business analytics and safety risk intelligence.',
    llm=groq_llm,
    verbose=True
)

writer = Agent(
    role='Pharma Communications Specialist',
    goal='Draft a professional LinkedIn post and executive email summary for {drug_name}.',
    backstory='You translate technical drug trial data into engaging, professional reports.',
    llm=groq_llm,
    verbose=True
)

# 6. DEFINE THE TASKS
research_task = Task(
    description="Find 3 recent clinical trial updates for {drug_name} regarding {topic}.",
    expected_output="A list of 3 updates with NCT IDs and direct source links.",
    agent=researcher
)

analysis_task = Task(
    description="Analyze the risks of {drug_name} updates on market entry and clinical timelines.",
    expected_output="A structured Risk Intelligence Brief in Markdown format.",
    agent=analyst,
    human_input=True, # PAUSES FOR YOUR FEEDBACK: Type 'Proceed' in the box to finish.
    output_file="drug_risk_report.md"
)

summary_task = Task(
    description="Create a LinkedIn post and a 3-bullet email summary for {drug_name}.",
    expected_output="A ready-to-use LinkedIn draft and professional email.",
    agent=writer,
    context=[analysis_task],
    output_file="final_communications.txt"
)

# 7. KICKOFF THE ENGINE
pri_crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[research_task, analysis_task, summary_task],
    process=Process.sequential
)

# CHANGE YOUR DRUG NAME AND TOPIC HERE!
print("🚀 Launching PRI Engine...")
result = pri_crew.kickoff(inputs={
    'drug_name': 'Pembrolizumab',
    'topic': 'Phase 3 Oncology trial status changes'
})

print("\n\n########################")
print("## FINAL PROJECT OUTPUT ##")
print("########################\n")
print(result)





  Using cached pydantic-2.11.10-py3-none-any.whl.metadata (68 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached tiktoken-0.8.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
  Using cached pydantic_core-2.33.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
Using cached pydantic-2.11.10-py3-none-any.whl (444 kB)
Using cached pydantic_core-2.33.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.0 MB)
Using cached python_dotenv-1.1.1-py3-none-any.whl (20 kB)
Using cached tiktoken-0.8.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.2 MB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.0.1
    Uninstalling python-dotenv-1.0.1:
      Successfully uninstalled python-dotenv-1.0.1
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uni

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Regulatory Researcher                                                                              │
│                                                                                                                 │
│  Task: Find 3 recent clinical trial updates for Pembrolizumab regarding Phase 3 Oncology trial status changes.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'Pembrolizumab Phase 3 Oncology trial status changes site:clinicaltrials.gov', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': '[PDF] 11-Dec-2020 A P...
Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'Pembrolizumab Phase 3 Oncology trial status changes updates site:clinicaltrials.gov OR site:fda.gov', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title'...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Regulatory Researcher                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. NCT05092360 - https://clinicaltrials.gov/study/nct05092360                                                  │
│  2. NCT05812807 - https://clinicaltrials.gov/study/NCT05812807                                                  │
│  3. NCT02853305 - https://clinicaltrials.gov/study/nct02853305                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Pharma Communications Specialist                                                                        │
│                                                                                                                 │
│  Task: Create a LinkedIn post and a 3-bullet email summary for Pembrolizumab.                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Pharma Communications Specialist                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **LinkedIn Post:**                                                                                             │
│                                                                                                                 │
│  Exciting Updates on Pembrolizumab: Navigating Risks for Long-Term Success                                      │
│                                                                                                                 │
│  As a Pharma Communications Specialist, I'm thrilled to share insights on Pembrolizumab, a monoclonal antibody  │
│  drug that has revolutionized the treatment of various cancers, including melanoma, lung cancer, and more.      │
│  With its mechanism of targeting the PD-1 receptor, Pembrolizumab has shown significant efficacy in clinical    │
│  trials.                                                                                                        │
│                                                                                                                 │
│  However, as with any groundbreaking treatment, there are risks associated with its market entry and clinical   │
│  timelines. Recent analysis of three clinical trials (NCT05092360, NCT05812807, and NCT02853305) highlights     │
│  the importance of understanding these risks to ensure the drug's long-term viability.                          │
│                                                                                                                 │
│  Key areas of focus include:                                                                                    │
│                                                                                                                 │
│  * Competition from other checkpoint inhibitors                                                                 │
│  * Regulatory environment and potential changes in guidelines                                                   │
│  * Pricing and reimbursement policies affecting access                                                          │
│                                                                                                                 │
│  By acknowledging and addressing these risks, stakeholders can work towards mitigating them and ensuring        │
│  Pembrolizumab remains a vital treatment option for patients.                                                   │
│                                                                                                                 │
│  Let's continue the conversation on how to navigate the complex landscape of cancer treatments and ensure       │
│  innovative therapies like Pembrolizumab reach those who need them most.                                        │
│                                                                                                                 │
│  #Pembrolizumab #CancerTreatment #Pharma #Innovation #Healthcare                                                │
│                                                                                                                 │
│  **Executive Email Summary:**                                                                                   │
│                                                                                                                 │
│  Subject: Pembrolizumab Market Entry and Clinical Timel

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'agent_execution_started' 
(expected 'crew_kickoff_started')



########################
## FINAL PROJECT OUTPUT ##
########################

**LinkedIn Post:**

Exciting Updates on Pembrolizumab: Navigating Risks for Long-Term Success

As a Pharma Communications Specialist, I'm thrilled to share insights on Pembrolizumab, a monoclonal antibody drug that has revolutionized the treatment of various cancers, including melanoma, lung cancer, and more. With its mechanism of targeting the PD-1 receptor, Pembrolizumab has shown significant efficacy in clinical trials.

However, as with any groundbreaking treatment, there are risks associated with its market entry and clinical timelines. Recent analysis of three clinical trials (NCT05092360, NCT05812807, and NCT02853305) highlights the importance of understanding these risks to ensure the drug's long-term viability.

Key areas of focus include:

* Competition from other checkpoint inhibitors
* Regulatory environment and potential changes in guidelines
* Pricing and reimbursement policies affecting acces

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯